# 2.06 HuggingFace / Sentence Transformers Embeddings

`HuggingFaceEmbeddings`는 **Hugging Face Hub의 모든 sentence-transformers 모델**을 로컬에서 실행한다.
Ollama와 달리 별도 데몬이 필요 없고, `sentence-transformers` 라이브러리가 직접 모델을 로드한다.
**API 키가 전혀 필요 없음** (로컬 다운로드) — 다만 첫 실행 시 모델 파일을 내려받는다.

## 학습 목표

- `HuggingFaceEmbeddings`로 로컬 sentence-transformers 모델을 호출한다
- `BAAI/bge-m3`의 다국어/한국어 품질을 체감한다
- `encode_kwargs`, `model_kwargs`로 정규화·디바이스 선택·배치 크기를 튜닝한다
- HuggingFace Inference API (`HuggingFaceEndpointEmbeddings`) 원격 경로의 차이를 구분한다

## 언제 쓰나

- **오픈소스 임베더 전체 카탈로그**에 접근해야 할 때 (BGE, E5, GTE, multilingual-e5 등)
- **한국어 특화 모델**(`BAAI/bge-m3`, `intfloat/multilingual-e5-large`)을 현업에 투입
- **GPU 기계**가 있어 대량 임베딩을 빠르게 인덱싱하고 싶을 때
- **키 관리 0**: 완전 로컬, 외부 API 호출 없음

## 2.06.1 환경 설정

필요 패키지: `langchain-huggingface`, `sentence-transformers`. 첫 실행 시 `BAAI/bge-m3` (~2.3GB)를 다운로드한다. 네트워크와 디스크 여유를 확인.

In [ ]:
# !pip install -U langchain langchain-huggingface sentence-transformers

from langchain_huggingface import HuggingFaceEmbeddings
HuggingFaceEmbeddings(model_name="BAAI/bge-m3").embed_query("ping")  # downloads model locally

## 2.06.2 기본 사용법

`model_name`은 Hugging Face Hub 경로. 로컬 캐시에 한 번 받으면 이후 호출은 네트워크를 쓰지 않는다.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

q_vec = embeddings.embed_query("BGE-M3은 어떤 모델인가?")
print("query dim:", len(q_vec), "| preview:", q_vec[:4])

docs = [
    "BGE-M3은 BAAI의 다국어 다기능 임베더다.",
    "Dense, sparse, ColBERT 세 가지 출력을 지원한다.",
    "최대 8192 토큰 입력을 받는다.",
]
doc_vecs = embeddings.embed_documents(docs)
print("docs:", len(doc_vecs), "x", len(doc_vecs[0]))

## 2.06.3 차원 · 비용 · 다국어 성능

| 모델 | 차원 | 최대 토큰 | 언어 | 메모 |
|------|------|----------|------|------|
| `BAAI/bge-m3` | 1024 | 8192 | 100+ | 권장 다국어 기본, 긴 문서 OK |
| `intfloat/multilingual-e5-large` | 1024 | 512 | 100+ | E5 패밀리 다국어 최상위 |
| `BAAI/bge-large-en-v1.5` | 1024 | 512 | 영어 | 영어 전용 고품질 |
| `sentence-transformers/all-MiniLM-L6-v2` | 384 | 256 | 영어 | 경량·저지연 |
| `jhgan/ko-sroberta-multitask` | 768 | 512 | 한국어 | 한국어 특화 |

- **비용**: 전력/시간만 — API 비용 없음
- **한국어**: `BAAI/bge-m3`가 범용 최상위. 한국어만 다루면 `jhgan/ko-sroberta-multitask`가 가볍다
- **속도**: GPU(CUDA/MPS) 권장. 대용량 인덱싱은 배치·정규화 튜닝이 결정적

## 2.06.4 벡터스토어 연동 미리보기

In [ ]:
# !pip install -U langchain-chroma
from langchain_chroma import Chroma

store = Chroma.from_texts(
    texts=docs,
    embedding=HuggingFaceEmbeddings(model_name="BAAI/bge-m3"),
    collection_name="demo_hf",
)
hits = store.similarity_search("다기능 임베더", k=2)
for h in hits:
    print("-", h.page_content)

## 2.06.5 로컬 최적화 — `encode_kwargs`, `model_kwargs`

`HuggingFaceEmbeddings`는 내부 `SentenceTransformer.encode(**encode_kwargs)`를 호출한다.

| 위치 | 키 | 용도 |
|------|-----|------|
| `model_kwargs` | `device` | `"cuda"`, `"mps"`, `"cpu"` 선택 |
| `model_kwargs` | `trust_remote_code` | 허브 커스텀 구현 허용 여부 |
| `encode_kwargs` | `normalize_embeddings` | True 시 L2 정규화 → 코사인 유사도 = 내적 |
| `encode_kwargs` | `batch_size` | 기본 32. GPU면 64~128 권장 |
| `encode_kwargs` | `show_progress_bar` | 대량 적재 시 True |

In [ ]:
tuned = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},  # GPU 있으면 "cuda" 또는 "mps"
    encode_kwargs={
        "normalize_embeddings": True,   # 정규화로 코사인 = 내적
        "batch_size": 16,
    },
)
v = tuned.embed_query("정규화된 임베딩")
# 정규화 여부 확인: L2 norm ≈ 1.0
import math
norm = math.sqrt(sum(x*x for x in v))
print(f"dim={len(v)} | L2 norm={norm:.4f}")

## 2.06.6 HuggingFace Inference Endpoint (원격)

모델 파일을 로컬로 내려받기 어렵거나 GPU가 없는 경우 HF Inference API / Endpoint 경유 호출이 가능하다.
이때는 `HuggingFaceEndpointEmbeddings` 를 사용하고 `HUGGINGFACEHUB_API_TOKEN`이 필요하다.

In [ ]:
# from langchain_huggingface import HuggingFaceEndpointEmbeddings
# remote = HuggingFaceEndpointEmbeddings(
#     model="BAAI/bge-m3",  # Inference API 엔드포인트 이름 또는 자체 Inference Endpoint URL
# )
# print(remote.embed_query("원격 HF 호출")[:3])
print("원격 HF Endpoint 섹션은 예시. HUGGINGFACEHUB_API_TOKEN이 있을 때만 활성화.")

## 체크리스트

- [ ] 첫 실행 시 모델 다운로드 시간/디스크를 감안(`BAAI/bge-m3`는 ~2.3GB)
- [ ] GPU가 있다면 `model_kwargs={'device': 'cuda'}` 또는 `'mps'`
- [ ] 코사인 유사도를 쓰는 vector store라면 `normalize_embeddings=True` 강제
- [ ] 한국어만 다루면 `jhgan/ko-sroberta-multitask`로 모델 경량화 검토
- [ ] 로컬 자원이 부족하면 `HuggingFaceEndpointEmbeddings` 원격 경로 전환

## 다음

- `../03_vectorstores/` — 로컬 HF 임베더를 Chroma/FAISS/Qdrant에 적재
- `../05_retrievers/` — BGE 임베더 + BM25 하이브리드 검색

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/text_embedding/huggingface
- sentence-transformers: https://www.sbert.net/
- BGE-M3 모델 카드: https://huggingface.co/BAAI/bge-m3